In [14]:
import os
import pandas as pd
import soundfile as sf
import time
from maad import sound
from maad.util import power2dB, format_features
from maad.rois import create_mask, select_rois
from maad.features import centroid_features
import numpy as np

In [15]:
std = 1.25
bin_std = 1.3
bin_per = 0.8

In [16]:
def process_audio(file_path, output_folder):
    print(f"Processing file: {file_path}")
    start_time = time.time()

    try:
        # Load the audio file
        s, fs = sound.load(file_path)
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        return pd.DataFrame()

    s_filt = sound.select_bandwidth(s, fs, fcut=100, forder=3, ftype='highpass')

    # Spectrogram parameters
    db_max = 70
    Sxx, tn, fn, ext = sound.spectrogram(s_filt, fs, nperseg=1024, noverlap=512)
    Sxx_db = power2dB(Sxx, db_range=db_max) + db_max

    # Background removal and smoothing
    Sxx_db_rmbg, _, _ = sound.remove_background(Sxx_db)
    Sxx_db_smooth = sound.smooth(Sxx_db_rmbg, std=std)
    im_mask = create_mask(im=Sxx_db_smooth, mode_bin='relative', bin_std=bin_std, bin_per=bin_per)
    im_rois, df_rois = select_rois(im_mask, min_roi=50, max_roi=None)

    if df_rois.empty:
        print(f"No ROIs found in file: {file_path}")
        return pd.DataFrame()

    # Format ROIs
    df_rois = format_features(df_rois, tn, fn)
    # Filter ROIs for those with centroid frequency below 2000Hz
    low_freq_rois = df_rois[df_rois['max_f'] <= 2000]
    print(low_freq_rois)

    if low_freq_rois.empty:
        print(f"No low frequency ROIs found in file: {file_path}")
        return pd.DataFrame()

    # Extract start and end times of the filtered ROIs
    low_freq_timestamps = low_freq_rois[['min_t', 'max_t', 'min_f', 'max_f']]
    low_freq_timestamps.columns = ['start_time', 'end_time', 'min_f', 'max_f']

    # Generate 5-second audio clips with detected region in the middle
    audio_clips = []
    clip_duration = 5.0  # seconds

    for i, (start_time, end_time, min_f, max_f) in enumerate(low_freq_timestamps.itertuples(index=False)):
        mid_point = (start_time + end_time) / 2
        window_start = max(0, mid_point - clip_duration / 2)
        window_end = window_start + clip_duration

        # Check if the clip is in the first 5 seconds or last 5 seconds of the audio
        if window_start < 0:
            window_start = 0
            window_end = clip_duration
        elif window_end > len(s) / fs:
            window_end = len(s) / fs
            window_start = window_end - clip_duration

        start_sample = int(window_start * fs)
        end_sample = int(window_end * fs)
        audio_clip = s[start_sample:end_sample]
        clip_filename = f'clip_{os.path.basename(file_path).split(".")[0]}_{i}.wav'
        clip_path = os.path.join(output_folder, clip_filename)
        sf.write(clip_path, audio_clip, fs)
        audio_clips.append((start_time, end_time, window_start, window_end, min_f, max_f, clip_filename))

    # Create DataFrame for the audio clips
    df_audio_clips = pd.DataFrame(audio_clips, columns=['start_time', 'end_time', 'window_start', 'window_end', 'min_f', 'max_f', 'audio_clip'])

    end_time = time.time()
    print(f"Finished processing file: {file_path}")

    return df_audio_clips


In [17]:
def process_folder(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    all_timestamps = []
    for filename in os.listdir(input_folder):
        if filename.endswith('.wav'):
            file_path = os.path.join(input_folder, filename)
            timestamps = process_audio(file_path, output_folder)
            all_timestamps.append(timestamps)

    return all_timestamps

# Example usage
input_folder = "D:/Aqoustics/Boat/Y_N_Boat/test"
output_folder = "D:/Aqoustics/Aqoustics-Unsupervised/data/output"
all_timestamps = process_folder(input_folder, output_folder)

# Save the timestamps to a CSV file
for i, timestamps in enumerate(all_timestamps):
    timestamps.to_csv(os.path.join(output_folder, f'timestamps_{i}.csv'), index=False)

Processing file: D:/Aqoustics/Boat/Y_N_Boat/test\y_051220_cargo.wav
   labelID    label  min_y  min_x  max_y  max_x   min_f    min_t    max_f  \
0        1  unknown      0   7827     13   8250    0.00  125.248   406.25   
1        2  unknown      0   8273     15   8591    0.00  132.384   468.75   
2        3  unknown      0   8875     18   9872    0.00  142.016   562.50   
3        4  unknown      0   9890     37  12935    0.00  158.256  1156.25   
4        5  unknown      1   2492      9   2514   31.25   39.888   281.25   
..     ...      ...    ...    ...    ...    ...     ...      ...      ...   
81     105  unknown      4   1486      9   1507  125.00   23.792   281.25   
82     109  unknown      4   1579     10   1595  125.00   25.280   312.50   
83     113  unknown      4   2695      9   2711  125.00   43.136   281.25   
84     116  unknown     12  12382     22  12393  375.00  198.128   687.50   
85     120  unknown     14  11892     24  11908  437.50  190.288   750.00   

      m

In [18]:
all_timestamps

[    start_time  end_time  window_start  window_end   min_f    max_f  \
 0      125.248   132.016       126.132     131.132    0.00   406.25   
 1      132.384   137.472       132.428     137.428    0.00   468.75   
 2      142.016   157.968       147.492     152.492    0.00   562.50   
 3      158.256   206.976       180.116     185.116    0.00  1156.25   
 4       39.888    40.240        37.564      42.564   31.25   281.25   
 ..         ...       ...           ...         ...     ...      ...   
 81      23.792    24.128        21.460      26.460  125.00   281.25   
 82      25.280    25.536        22.908      27.908  125.00   312.50   
 83      43.136    43.392        40.764      45.764  125.00   281.25   
 84     198.128   198.304       195.716     200.716  375.00   687.50   
 85     190.288   190.544       187.916     192.916  437.50   750.00   
 
                     audio_clip  
 0    clip_y_051220_cargo_0.wav  
 1    clip_y_051220_cargo_1.wav  
 2    clip_y_051220_cargo_2.wav 

In [22]:
# Access the DataFrame from the list
df = all_timestamps[0]

# Now filter the DataFrame for the specific audio clip
filtered_timestamps = df[df['audio_clip'] == 'clip_y_051220_cargo_3.wav']

# This will give you the rows corresponding to the specified audio clip.
print(filtered_timestamps)


   start_time  end_time  window_start  window_end  min_f    max_f  \
3     158.256   206.976       180.116     185.116    0.0  1156.25   

                  audio_clip  
3  clip_y_051220_cargo_3.wav  
